## Sentiment Analysis

In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize,sent_tokenize 
from nltk.corpus import stopwords
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , classification_report
import pickle
import sklearn
import numpy as np

In [2]:
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NLTK:", nltk.__version__)
print("NumPy:" , np.__version__)

Pandas: 2.2.3
Scikit-learn: 1.6.0
NLTK: 3.9.1
NumPy: 1.26.4


In [3]:
data = pd.read_csv("IMDB Dataset.csv")

In [4]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
duplicate_count = data.duplicated().sum()

In [6]:
print(duplicate_count)

418


In [7]:
data = data.drop_duplicates().reset_index(drop=True)

In [8]:
duplicate_count = data.duplicated().sum()

In [9]:
print(duplicate_count)

0


In [10]:
Missing_value = data.isnull().sum()

In [11]:
print(Missing_value)

review       0
sentiment    0
dtype: int64


In [12]:
data["lowercase"] = data["review"].str.lower()

In [13]:
data.head()

,review,sentiment,lowercase
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is..."


In [14]:
data["html_remove"] = data["lowercase"].apply(lambda review : re.sub("<.*?>" , "", review ))

In [15]:
data["url_remove"] = data["html_remove"].apply(lambda x : re.sub (r"https?://\S+|www\.\S+" , "", x ))

In [16]:
data["punc_remove"] = data["url_remove"].apply(lambda review: re.sub(r"[^a-zA-Z\s]", "", review ))

In [17]:
data.head()

,review,sentiment,lowercase,html_remove,url_remove,punc_remove
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,a wonderful little production. the filming tec...,a wonderful little production. the filming tec...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...


In [18]:
data["word_tokenize"] = data["punc_remove"].apply(lambda review : [word for word in  review.split()])

In [19]:
data.head()

,review,sentiment,lowercase,html_remove,url_remove,punc_remove,word_tokenize
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,a wonderful little production. the filming tec...,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin..."
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon..."


In [20]:
no_stopword = stopwords.words("english")

In [21]:
data["no_stopword"] = data["word_tokenize"].apply(lambda review : [word for word in review if word not in no_stopword])

In [22]:
data.head()

,review,sentiment,lowercase,html_remove,url_remove,punc_remove,word_tokenize,no_stopword
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione...","[one, reviewers, mentioned, watching, oz, epis..."
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,a wonderful little production. the filming tec...,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin...","[wonderful, little, production, filming, techn..."
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,...","[thought, wonderful, way, spend, time, hot, su..."
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl...","[basically, theres, family, little, boy, jake,..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon...","[petter, matteis, love, time, money, visually,..."


In [23]:
lemmatizer =  WordNetLemmatizer()

In [44]:
data["lemmatization"] = data["no_stopword"].apply(lambda review : " ".join(lemmatizer.lemmatize(word)for word in review))

In [25]:
data.head()

,review,sentiment,lowercase,html_remove,url_remove,punc_remove,word_tokenize,no_stopword,lemmatization
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione...","[one, reviewers, mentioned, watching, oz, epis...",one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. <br /><br />the...,a wonderful little production. the filming tec...,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin...","[wonderful, little, production, filming, techn...",wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,...","[thought, wonderful, way, spend, time, hot, su...",thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically there's a family where a little boy ...,basically theres a family where a little boy j...,"[basically, theres, a, family, where, a, littl...","[basically, theres, family, little, boy, jake,...",basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,"[petter, matteis, love, in, the, time, of, mon...","[petter, matteis, love, time, money, visually,...",petter matteis love time money visually stunni...


In [26]:
data["clean_text"]  = data["lemmatization"]

In [27]:
data["sentiment"] = data["sentiment"].map({"negative" : 0 , "positive" : 1} )

In [28]:
bow = CountVectorizer() 

In [29]:
fit_transform = bow.fit_transform(data["lemmatization"][4000:6000])

In [30]:
bow_df = pd.DataFrame(fit_transform.toarray(),columns=bow.get_feature_names_out())

In [31]:
print(bow_df)

      aaall  aaaugh  aag  aaron  ab  abandon  abandoned  abandoning  abashed  \
0         0       0    0      0   0        0          0           0        0   
1         0       0    0      0   0        0          0           0        0   
2         0       0    0      0   0        0          0           0        0   
3         0       0    0      0   0        0          0           0        0   
4         0       0    0      0   0        0          0           0        0   
...     ...     ...  ...    ...  ..      ...        ...         ...      ...   
1995      0       0    0      0   0        0          0           0        0   
1996      0       0    0      0   0        0          0           0        0   
1997      0       0    0      0   0        0          0           0        0   
1998      0       0    0      0   0        0          0           0        0   
1999      0       0    0      0   0        0          0           0        0   

      abbey  ...  zosh  zozo  zucco  zu

In [32]:
x = fit_transform
y = data["sentiment"][4000:6000]

In [33]:
x_train , x_test , y_train , y_test = train_test_split(x,y ,test_size = 0.3 , random_state=1)

In [34]:
model = LogisticRegression(random_state = 1 , max_iter = 1000) .fit(x_train,y_train)

In [35]:
xy_predict = model.predict(x_test)

In [36]:
accuracy_score(y_test , xy_predict)

0.795

In [37]:
print(classification_report(y_test,xy_predict))

              precision    recall  f1-score   support

           0       0.78      0.80      0.79       289
           1       0.81      0.79      0.80       311

    accuracy                           0.80       600
   macro avg       0.79      0.80      0.79       600
weighted avg       0.80      0.80      0.80       600



In [38]:
review = ["This movie was absolutely amazing and I loved it"]

In [39]:
vector = bow.transform(review)

In [40]:
prediction = model.predict(vector)[0]

In [41]:
if prediction == 1:
    print("positive sentiment")
else:    
    print("negative sentiment")

positive sentiment


In [42]:
with open("model","wb") as file:
    pickle.dump(model , file)
with open("bow" , "wb") as file:
    pickle.dump(bow , file)